In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

data_path = '/content/drive/MyDrive/credit_scoring/data/'

# Confirm what's in the folder
import os
os.listdir(data_path)

['application_train.csv',
 'bureau.csv',
 'installments_payments.csv',
 'previous_application.csv',
 'applicant_profile_202607011234.csv']

In [ ]:
import pandas as pd

data_path = '/content/drive/MyDrive/credit_scoring/data/'

# Load the three files we need — using the exact timestamped name
profile = pd.read_csv(data_path + 'applicant_profile_202607011234.csv')
installments = pd.read_csv(data_path + 'installments_payments.csv')
app = pd.read_csv(data_path + 'application_train.csv')

print("Profile shape:", profile.shape)
print("Installments shape:", installments.shape)
print("App shape:", app.shape)

profile.head()

Profile shape: (307511, 3)
Installments shape: (13605401, 8)
App shape: (307511, 122)


,SK_ID_CURR,TARGET,profile
0,100002,1,Has bureau history
1,100003,0,Has bureau history
2,100004,0,Has bureau history
3,100006,0,No bureau history
4,100007,0,No bureau history


In [ ]:
profile['profile'].value_counts()

,count
profile,
Has bureau history,177066
No bureau history,89634
Thin file (1 entry),40811


In [ ]:
# TEST group = thin-file and no-bureau people
test_ids = profile[profile['profile'].isin(['No bureau history', 'Thin file (1 entry)'])]['SK_ID_CURR']

# TRAIN group = everyone else (people WITH bureau history)
train_ids = profile[profile['profile'] == 'Has bureau history']['SK_ID_CURR']

print("Train applicants (has bureau):", len(train_ids))
print("Test applicants (thin/no bureau):", len(test_ids))

Train applicants (has bureau): 177066
Test applicants (thin/no bureau): 130445


In [ ]:
# Flag each payment - on time or not
installments['paid_on_time'] = (
    installments['DAYS_ENTRY_PAYMENT'] <= installments['DAYS_INSTALMENT'] + 7
).astype(int)

# How many days late - each payment
installments['days_late'] = installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']

# Payment behaviour summary - per applicant
payment = installments.groupby('SK_ID_CURR').agg(
    pct_on_time=('paid_on_time', 'mean'),
    avg_days_late=('days_late', 'mean'),
    num_late_payments=('paid_on_time', lambda x: (x == 0).sum())
).reset_index()

print("Payment features shape:", payment.shape)
payment.head()

Payment features shape: (339587, 4)


,SK_ID_CURR,pct_on_time,avg_days_late,num_late_payments
0,100001,0.857143,-7.285714,1
1,100002,1.000000,-20.421053,0
2,100003,1.000000,-7.160000,0
3,100004,1.000000,-7.666667,0
4,100005,1.000000,-23.555556,0


In [ ]:
# Master table: each applicant's payment behaviour + their TARGET + their profile
master = profile.merge(payment, on='SK_ID_CURR', how='inner')

print("Master shape:", master.shape)
print("Profiles present:")
print(master['profile'].value_counts())

Master shape: (291643, 6)
Profiles present:
profile
Has bureau history     168855
No bureau history       84587
Thin file (1 entry)     38201
Name: count, dtype: int64


In [ ]:
# Split master into train (has bureau) and test (thin/no bureau)
train_df = master[master['profile'] == 'Has bureau history']
test_df = master[master['profile'].isin(['No bureau history', 'Thin file (1 entry)'])]

# Features = payment behaviour only. Answer = TARGET.
feature_cols = ['pct_on_time', 'avg_days_late', 'num_late_payments']

X_train = train_df[feature_cols]
y_train = train_df['TARGET']

X_test = test_df[feature_cols]
y_test = test_df['TARGET']

print("Train:", X_train.shape, "| Default rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape, "| Default rate:", round(y_test.mean(), 3))

Train: (168855, 3) | Default rate: 0.077
Test:  (122788, 3) | Default rate: 0.088


In [ ]:
# Check missing values before training
print(X_train.isnull().sum())
print(X_test.isnull().sum())

pct_on_time          0
avg_days_late        1
num_late_payments    0
dtype: int64
pct_on_time          0
avg_days_late        7
num_late_payments    0
dtype: int64


In [ ]:
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("Train missing:", X_train.isnull().sum().sum())
print("Test missing:", X_test.isnull().sum().sum())

Train missing: 0
Test missing: 0


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Scale — fit on train only, apply to both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train on has-bureau people
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# Test on thin-file/no-bureau people the model NEVER saw
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

print("AUC on held-out thin-file/no-bureau group:", round(auc, 3))

AUC on held-out thin-file/no-bureau group: 0.561


In [ ]:
master['payer_type'] = master['pct_on_time'].apply(
    lambda x: 'On-time payer' if x >= 0.95 else 'Late payer'
)
master['group'] = master['profile'].apply(
    lambda x: 'Has bureau' if x == 'Has bureau history' else 'Thin/No bureau'
)
result = master.groupby(['group', 'payer_type'])['TARGET'].mean().round(3) * 100
print(result)

group           payer_type   
Has bureau      Late payer       10.2
                On-time payer     7.3
Thin/No bureau  Late payer       13.4
                On-time payer     8.1
Name: TARGET, dtype: float64
